# Inicio del proyecto con PLAsTiCC Ia II Ibc

Resúmenes de curvas de luz; investiguen la identificación de Ibc frente a Ia y II.

Este notebook carga los datos y explica su estructura. **No es una solución del proyecto.** Completen las secciones de investigación del final. Trabajen en Colab gratuito.

Fuente: https://zenodo.org/records/2539456

## 1. Cargar los datos
Descompriman la entrega en su computador y abran este `.ipynb` en Colab. Ejecuten la siguiente celda y suban **ML_2026_11_10_PLAsTiCC_Ia_II_Ibc_DATOS.zip**, ubicado junto a este notebook. No suban el paquete general ni vuelvan a descargar la fuente original. La carga no necesita GPU.

In [1]:
from pathlib import Path
import io
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

archivo = Path('ML_2026_11_10_PLAsTiCC_Ia_II_Ibc_DATOS.zip')
if not archivo.exists():
    from google.colab import files
    subidos = files.upload()
    if archivo.name not in subidos:
        raise ValueError("Sube el ZIP de DATOS que indica este notebook, con su nombre original.")

paquete = zipfile.ZipFile(archivo)
def leer_auxiliar(nombre):
    """Devuelve bytes de un archivo auxiliar incluido en este ZIP."""
    return io.BytesIO(paquete.read(nombre))

datos = np.load(leer_auxiliar('ML_2026_11_10_PLAsTiCC_Ia_II_Ibc_arrays.npz'), allow_pickle=False)
X, y = datos['X'], datos['y']
ids, split, nombres = datos['ids'], datos['split'], datos['names']
print('X:', X.shape, X.dtype, '| y:', y.shape)
print(dict(enumerate(nombres)))

X: (3990, 72) float32 | y: (3990,)
{0: np.str_('SN Ia'), 1: np.str_('SN II'), 2: np.str_('SN Ibc')}


## 2. Respetar las particiones
Cada fila de `X`, `y`, `ids` y `split` corresponde a la misma observación. `y` es una etiqueta entera desde cero. Los identificadores y las etiquetas no son predictores. Usen `train` para aprender parámetros y transformaciones, `val` para elegir modelos y `test` para la evaluación final. La tabla de tamaños no evalúa ningún modelo.

In [7]:
# Identificar los ejemplos de cada partición
train = split == 'train'
val = split == 'val'
test = split == 'test'

# Separar las características
X_train = X[train]
X_val = X[val]
X_test = X[test]

# Separar las etiquetas
y_train = y[train]
y_val = y[val]
y_test = y[test]

# Mostrar cuántos ejemplos tiene cada conjunto
print("Entrenamiento:", len(y_train))
print("Validación:", len(y_val))
print("Test:", len(y_test))

Entrenamiento: 2394
Validación: 798
Test: 798


## 3. Escalar sin fuga de información
Las características ya están calculadas. El ejemplo ajusta un escalador **solo con entrenamiento** y transforma después validación y test. No vuelvan a ajustar el escalador sobre el conjunto completo. Para cada representación alternativa ajusten su propio escalador de entrenamiento. Los árboles no requieren este escalado.

In [3]:
from sklearn.preprocessing import StandardScaler
escalador = StandardScaler()
X_train_red = escalador.fit_transform(X_train).astype('float32')
X_val_red = escalador.transform(X_val).astype('float32')
X_test_red = escalador.transform(X_test).astype('float32')
print('Entrada preparada para MLP:', X_train_red.shape)

Entrada preparada para MLP: (2394, 72)


### Qué representan las 72 columnas
En cada banda (0 a 5), el orden es: número de observaciones, presencia de datos, mediana del flujo, percentil 90 menos 10, percentil 75 menos 25, desviación estándar, mínimo, máximo, mediana del error, máximo flujo/error, fracción con flujo/error > 5 y duración observada. Son 12 × 6 = 72 variables. Una banda ausente tiene ceros y su indicador de presencia es cero. Conserven los flujos negativos.

### Recalcular con menos observaciones
La función siguiente se proporciona para que puedan estudiar cobertura. No usa etiquetas ni aprende sobre test. Conserva la identidad y partición de los objetos. Una fracción menor selecciona menos observaciones por banda; no equivale a eliminar variables ni a observar solo el inicio de la curva. Ejecutar la última celda opcional puede tardar unos minutos.

In [4]:
"""Features observacionales. No usan etiquetas, redshift ni información simulada latente."""

def plasticc_features(curves,ids,fraction=1.,seed=2026):
 """12 features por banda. Submuestreo aleatorio por objeto/banda, antes de resumir.
 Una misma permutación produce subconjuntos anidados para distintas fracciones.
 No equivale a clasificación temprana. count y mask exponen la ausencia de datos.
 """
 if not 0<fraction<=1:raise ValueError('fraction debe pertenecer a (0,1]')
 groups={k:v for k,v in curves.groupby('object_id',sort=False)}
 out=[]
 for oid in ids:
  obj=groups[int(oid)];features=[]
  for band in range(6):
   b=obj[obj.passband==band].sort_values('mjd')
   rng=np.random.default_rng(np.random.SeedSequence([seed,int(oid),band]))
   n=int(np.floor(len(b)*fraction)); b=b.iloc[rng.permutation(len(b))[:n]].sort_values('mjd')
   if n==0:features.extend([0.]*12);continue
   f=b.flux.to_numpy();e=b.flux_err.to_numpy();t=b.mjd.to_numpy()
   q=np.quantile(f,[.1,.25,.5,.75,.9]);snr=f/e
   features.extend([n,1.,q[2],q[4]-q[0],q[3]-q[1],np.std(f),np.min(f),np.max(f),np.median(e),np.max(snr),np.mean(snr>5),np.ptp(t)])
  out.append(features)
 return np.asarray(out,dtype='float32')


In [5]:
observaciones = pd.read_csv(leer_auxiliar('ML_2026_11_10_PLAsTiCC_Ia_II_Ibc_plasticc_observaciones.csv.gz'), compression='gzip')
print(observaciones.shape)
print(observaciones.head())
# Opcional: descomenten para construir otra condición experimental.
# X_mitad = plasticc_features(observaciones, ids, fraction=0.5, seed=2026)
# Luego separen con train / val / test y ajusten un NUEVO escalador solo en train.
# X_completo = plasticc_features(observaciones, ids, fraction=1.0)
# np.testing.assert_allclose(X_completo, X, rtol=1e-5, atol=1e-5)

(764572, 6)
   object_id         mjd  passband       flux  flux_err  detected_bool
0        730  59798.3205         2   1.177371  1.364300              0
1        730  59798.3281         1   2.320849  1.159247              0
2        730  59798.3357         3   2.939447  1.771328              0
3        730  59798.3466         4   2.128097  2.610659              0
4        730  59798.3576         5 -12.809639  5.380097              0


In [6]:
X_train

array([[ 7.2000000e+01,  1.0000000e+00,  2.4093000e-02, ...,
         8.5727015e+00,  7.8431375e-02,  8.5380841e+02],
       [ 7.2000000e+01,  1.0000000e+00,  1.0567141e+00, ...,
         1.9283287e+01,  9.0909094e-02,  8.5366827e+02],
       [ 6.3000000e+01,  1.0000000e+00, -2.5813800e-01, ...,
         4.6116304e+00,  0.0000000e+00,  8.7176971e+02],
       ...,
       [ 1.3000000e+01,  1.0000000e+00,  3.1267440e+00, ...,
         7.2952676e+00,  2.7027028e-02,  8.8671729e+02],
       [ 1.3000000e+01,  1.0000000e+00,  3.1519001e+00, ...,
         7.9522133e+00,  9.0909094e-02,  1.0670073e+03],
       [ 1.2000000e+01,  1.0000000e+00,  3.5233209e+00, ...,
         3.3934116e+00,  0.0000000e+00,  1.0329121e+03]],
      shape=(2394, 72), dtype=float32)

## 4. Desarrollar la investigación
Completen las siguientes secciones con texto, código y resultados. Consulten el enunciado y las rúbricas. No es necesario aplicar todas las ideas propuestas. Seleccionen una pregunta y una comparación que puedan justificar.

## Pregunta e hipótesis
¿Qué quieren averiguar y qué cambio observable esperan?

## Exploración y decisiones
¿Qué evidencia justifica su representación, preprocesamiento y métricas?

## Modelo de referencia y red neuronal
Implementen y justifiquen ambos modelos. Registren arquitectura, hiperparámetros, semillas, tiempos e historia de entrenamiento.

## Comparación controlada
Definan el factor que cambia y lo que mantienen fijo. Seleccionen con validación; repitan la comparación y comuniquen variabilidad.

## Evaluación final y errores
Evalúen condiciones predefinidas en test. Incluyan macro F1, matriz de confusión, pureza y completitud por clase, y ejemplos de errores.

## Conclusiones y fuentes
Respondan la pregunta con evidencia, limitaciones y alcance. Declaren fuentes, código reutilizado y apoyos utilizados.